In [5]:
import os
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260606_054312"

fills = pd.read_parquet(os.path.join(folder_path, "fills.parquet"))

fills

,ts,side,price,price_tick,qty,is_maker,fill_type,fill_status,inventory,mid,...,queue_ahead_at_join,snap_mid,markout_100ms,adverse_100ms,markout_500ms,adverse_500ms,markout_1000ms,adverse_1000ms,markout_5000ms,adverse_5000ms
0,1780713727579,BUY,60980.05,6098005,0.00017,True,BID_HIT,PARTIAL,0.000170,60980.065,...,0.00107,60980.065,-6.055,6.055,-6.055,6.055,-9.965,9.965,-20.045,20.045
1,1780713727579,BUY,60980.05,6098005,0.00009,True,BID_HIT,PARTIAL,0.000260,60980.065,...,0.00107,60980.065,-6.055,6.055,-6.055,6.055,-9.965,9.965,-20.045,20.045
2,1780713728540,BUY,60973.98,6097398,0.00009,True,BID_HIT,PARTIAL,0.000350,60973.995,...,0.00081,60973.995,-3.895,3.895,-3.895,3.895,-6.115,6.115,-20.935,20.935
3,1780713729615,BUY,60967.85,6096785,0.00009,True,BID_HIT,PARTIAL,0.000440,60967.865,...,0.00000,60967.865,-3.845,3.845,-3.845,3.845,-3.845,3.845,-14.805,14.805
4,1780713729615,BUY,60967.85,6096785,0.00009,True,BID_HIT,PARTIAL,0.000530,60967.865,...,0.00000,60967.865,-3.845,3.845,-3.845,3.845,-3.845,3.845,-14.805,14.805
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9287,1780715775115,BUY,60884.80,6088480,0.00009,True,BID_HIT,PARTIAL,1.455355,60884.815,...,0.00108,60884.815,-1.365,1.365,-1.365,1.365,-1.365,1.365,-1.365,1.365
9288,1780715775115,BUY,60884.80,6088480,0.00009,True,BID_HIT,PARTIAL,1.455445,60884.815,...,0.00108,60884.815,-1.365,1.365,-1.365,1.365,-1.365,1.365,-1.365,1.365
9289,1780715775115,BUY,60884.80,6088480,0.00009,True,BID_HIT,PARTIAL,1.455535,60884.815,...,0.00108,60884.815,-1.365,1.365,-1.365,1.365,-1.365,1.365,-1.365,1.365
9290,1780715775115,BUY,60884.80,6088480,0.00009,True,BID_HIT,PARTIAL,1.455625,60884.815,...,0.00108,60884.815,-1.365,1.365,-1.365,1.365,-1.365,1.365,-1.365,1.365


In [ ]:
"""
model = predict adverse selection

                ┌──────────────┐
market ───────► │ regime model │ ───► policy params
                └──────┬───────┘
                       │
                       ▼
        ┌──────────────────────────┐
        │ alpha / fair value stack │ ───► center
        └──────────┬───────────────┘
                   │
                   ▼
        ┌──────────────────────────┐
        │ toxicity model           │ ───► risk modifiers
        └──────────┬───────────────┘
                   │
                   ▼
             execution engine

So your best toxicity model is actually:

E[future markout loss | fill now]

That is the true label your XGBoost model should learn.

Not classification. Not heuristic imbalance.
"""

df = fills

horizons = [100, 500, 1000, 5000]

feature_cols = [
    "microprice_dev",
    "order_imbalance",
    "trade_imbalance",
    "volatility",
    "spread",
    "queue_ahead_bid",
    "queue_ahead_ask",
]

df = df.dropna()

split = int(len(df) * 0.8)
train = df.iloc[:split]
test = df.iloc[split:]

X_train = train[feature_cols].to_numpy(dtype=np.float32)
X_test = test[feature_cols].to_numpy(dtype=np.float32)

In [7]:
def ic(pred, y):
    return np.corrcoef(pred, y)[0, 1]

def rank_ic(pred, y):
    return spearmanr(pred, y).statistic

results = {}
models = {}

for h in horizons:

    y_train = train[f"markout_{h}ms"]
    y_test = test[f"markout_{h}ms"]

    model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    tox_std = np.std(train_pred)
    tox_mean_abs = np.mean(np.abs(train_pred))

    pred = model.predict(X_test)
    true = y_test.values

    # -------------------------
    # signal quality
    # -------------------------
    residual_ic = ic(pred, true)
    residual_rank_ic = rank_ic(pred, true)

    # -------------------------
    # direction accuracy
    # -------------------------
    hit_rate = (np.sign(pred) == np.sign(true)).mean()

    # -------------------------
    # true economic interpretation
    # -------------------------
    pnl_proxy = np.mean(pred * true)
    pnl_std = np.std(pred * true) + 1e-9
    sharpe_proxy = pnl_proxy / pnl_std

    models[h] = {
        "model": model,
        "tox_std": tox_std,
        "tox_mean_abs": tox_mean_abs
    }
    results[h] = {
        "Residual_IC": residual_ic,
        "Residual_Rank_IC": residual_rank_ic,
        "HitRate": hit_rate,
        "PnLProxy": pnl_proxy,
        "SharpeProxy": sharpe_proxy,
    }

results_df = pd.DataFrame(results)
results_df

,100,500,1000,5000
Residual_IC,0.364784,0.269746,0.287165,0.151535
Residual_Rank_IC,0.431182,0.377428,0.383253,0.152754
HitRate,0.763852,0.742873,0.739645,0.562668
PnLProxy,7.684049,9.249838,12.390385,14.550684
SharpeProxy,0.395406,0.334345,0.340654,0.138478


In [19]:
"""
SharpeProxy increasing with horizon

This is actually the key signal:

100ms → weak economic signal
500-1000ms → stronger signal

This tells you:

toxicity is not instantaneous micro-noise, it is short-horizon information flow

That's exactly what informed order flow looks like.

3. The important conceptual insight

You are NOT building:

a price predictor

You ARE building:

a liquidity filter

This model answers:

“Should I provide liquidity here or not?”

"""

"\nSharpeProxy increasing with horizon\n\nThis is actually the key signal:\n\n100ms → weak economic signal\n500-1000ms → stronger signal\n\nThis tells you:\n\ntoxicity is not instantaneous micro-noise, it is short-horizon information flow\n\nThat's exactly what informed order flow looks like.\n\n3. The important conceptual insight\n\nYou are NOT building:\n\na price predictor\n\nYou ARE building:\n\na liquidity filter\n\nThis model answers:\n\n“Should I provide liquidity here or not?”\n\n"

In [ ]:
artifact = {
    "model": models[100]["model"],
    "feature_cols": feature_cols,
    "target": "markout_100ms",
    "horizon_ms": 100,
}

joblib.dump(artifact, "data/toxicity_model.pkl")

['data/toxicity_model.pkl']